# M04-02 — KPIs

Referencia de validación. El alumno trabaja en `notebooks/alumno/M04-02-kpis.ipynb`.


## Celda 0 — localizar el repo


In [ ]:
import sys
from pathlib import Path

_here = Path.cwd().resolve()
ROOT = next(
    p
    for p in [_here, *_here.parents]
    if (p / "labs" / "_shared" / "session.py").is_file()
)
sys.path.insert(0, str(ROOT / "labs" / "_shared"))

from paths import RAW, STAGING, CURATED
from session import get_spark

print("ROOT   ", ROOT)
print("RAW    ", RAW, "existe:", RAW.is_dir())
print("STAGING", STAGING)
print("CURATED", CURATED)


In [ ]:
from pyspark.sql.functions import col, sum as fsum, countDistinct, round as fround, avg
spark = get_spark("novashop-m04")
fact = spark.read.parquet(str(STAGING / "fact_lines"))
customers = spark.read.parquet(str(STAGING / "customers_clean"))
sales = fact.join(customers, "customer_id", "inner").where(col("is_billable"))
print("sales", sales.count())
assert sales.count() == 1122
kpis = sales.agg(fround(fsum("gmv_line"), 2).alias("gmv"), countDistinct("order_id").alias("orders"))
kpis = kpis.withColumn("aov", fround(col("gmv") / col("orders"), 2))
kpis.show()
row = kpis.collect()[0]
assert row["orders"] == 469
assert 399000 < float(row["gmv"]) < 402000
orders = spark.read.parquet(str(STAGING / "orders_clean"))
ord_ok = orders.join(customers, "customer_id", "inner")
cancel = ord_ok.agg(avg((col("status") == "cancelled").cast("double")).alias("cancel_rate"))
cancel.show()
(
    sales.groupBy("channel_norm")
    .agg(fround(fsum("gmv_line"), 2).alias("gmv"), countDistinct("order_id").alias("orders"))
    .orderBy(col("gmv").desc())
    .show()
)
print("M04-02 OK")
